# 06 Decode Estimator

This notebook introduces a simple simulation for decode time and total latency.

Decode is the phase where the model generates output tokens one token at a time.

Prefill depends mostly on input prompt tokens.
Decode depends mostly on generated output tokens.

## Simulation Formula

This step is only a simulation, not a real benchmark.

We use:
- output token budgets: `128, 256, 512, 1024, 2048`
- tokens per second: `10, 25, 50, 100`

Formula:

`estimated_decode_time_seconds = expected_output_tokens / tokens_per_second`

If the prefill report exists, we also compute:

`estimated_total_latency_seconds = estimated_ttft_seconds + estimated_decode_time_seconds`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from decode_estimator import (
    DECODE_REPORT_PATH,
    FIGURES_DIR,
    PREFILL_REPORT_PATH,
    build_decode_report,
    build_decode_summary,
    create_decode_graphs,
    load_prefill_report,
    save_decode_report,
)

print("Project root:", PROJECT_ROOT)
print("Prefill report path:", PREFILL_REPORT_PATH)
print("Decode report path:", DECODE_REPORT_PATH)
print("Figures directory:", FIGURES_DIR)

Project root: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference
Prefill report path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/prefill_estimation_report.csv
Decode report path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/decode_estimation_report.csv
Figures directory: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/figures


## Load Prefill Report

We reuse the Step 5 prefill report so the decode simulation can combine TTFT and output generation time.

In [2]:
prefill_df = load_prefill_report()
prefill_df.head()

,scenario_name,tokenizer_name,context_window,reserved_output_tokens,prompt_tokens,prefill_load,base_ttft_ms,prefill_time_per_1k_tokens_ms,estimated_ttft_ms,estimated_ttft_seconds,interpretation
0,packed_prompt_8192,TinyLlama/TinyLlama-1.1B-Chat-v1.0,8192,512,3877,medium,100,40,255.08,0.2551,"Using packed prompt tokens=3877, this scenario..."
1,packed_prompt_32768,TinyLlama/TinyLlama-1.1B-Chat-v1.0,32768,512,3877,medium,100,40,255.08,0.2551,"Using packed prompt tokens=3877, this scenario..."
2,packed_prompt_8192,mistralai/Mistral-7B-Instruct-v0.2,8192,512,3836,medium,100,40,253.44,0.2534,"Using packed prompt tokens=3836, this scenario..."
3,packed_prompt_32768,mistralai/Mistral-7B-Instruct-v0.2,32768,512,3836,medium,100,40,253.44,0.2534,"Using packed prompt tokens=3836, this scenario..."
4,packed_prompt_4096,Qwen/Qwen2.5-1.5B-Instruct,4096,512,3371,medium,100,40,234.84,0.2348,"Using packed prompt tokens=3371, this scenario..."


## Build the Decode Estimation Report

For each row, we calculate:
- expected output tokens
- tokens per second
- estimated decode time
- estimated total latency
- a decode load label
- a short interpretation string

In [3]:
df = build_decode_report(prefill_df)
df = save_decode_report(df, DECODE_REPORT_PATH)
df.head(10)

,scenario_name,tokenizer_name,context_window,prompt_tokens,estimated_ttft_seconds,expected_output_tokens,tokens_per_second,estimated_decode_time_seconds,estimated_total_latency_seconds,decode_load,interpretation
16,packed_prompt_8192,TinyLlama/TinyLlama-1.1B-Chat-v1.0,8192,3877,0.2551,2048,10,204.8,205.0551,high,"With prompt_tokens=3877, estimated TTFT is abo..."
36,packed_prompt_32768,TinyLlama/TinyLlama-1.1B-Chat-v1.0,32768,3877,0.2551,2048,10,204.8,205.0551,high,"With prompt_tokens=3877, estimated TTFT is abo..."
76,packed_prompt_32768,mistralai/Mistral-7B-Instruct-v0.2,32768,3836,0.2534,2048,10,204.8,205.0534,high,"With prompt_tokens=3836, estimated TTFT is abo..."
56,packed_prompt_8192,mistralai/Mistral-7B-Instruct-v0.2,8192,3836,0.2534,2048,10,204.8,205.0534,high,"With prompt_tokens=3836, estimated TTFT is abo..."
116,packed_prompt_8192,Qwen/Qwen2.5-1.5B-Instruct,8192,3371,0.2348,2048,10,204.8,205.0348,high,"With prompt_tokens=3371, estimated TTFT is abo..."
96,packed_prompt_4096,Qwen/Qwen2.5-1.5B-Instruct,4096,3371,0.2348,2048,10,204.8,205.0348,high,"With prompt_tokens=3371, estimated TTFT is abo..."
136,packed_prompt_32768,Qwen/Qwen2.5-1.5B-Instruct,32768,3371,0.2348,2048,10,204.8,205.0348,high,"With prompt_tokens=3371, estimated TTFT is abo..."
156,packed_prompt_4096,TinyLlama/TinyLlama-1.1B-Chat-v1.0,4096,2663,0.2065,2048,10,204.8,205.0065,high,"With prompt_tokens=2663, estimated TTFT is abo..."
176,packed_prompt_4096,mistralai/Mistral-7B-Instruct-v0.2,4096,2621,0.2048,2048,10,204.8,205.0048,high,"With prompt_tokens=2621, estimated TTFT is abo..."
12,packed_prompt_8192,TinyLlama/TinyLlama-1.1B-Chat-v1.0,8192,3877,0.2551,1024,10,102.4,102.6551,high,"With prompt_tokens=3877, estimated TTFT is abo..."


## Full Report Sorted by Estimated Total Latency

The highest rows represent the heaviest decode scenarios, especially when output budgets are large and tokens/sec is low.

In [4]:
df

,scenario_name,tokenizer_name,context_window,prompt_tokens,estimated_ttft_seconds,expected_output_tokens,tokens_per_second,estimated_decode_time_seconds,estimated_total_latency_seconds,decode_load,interpretation
16,packed_prompt_8192,TinyLlama/TinyLlama-1.1B-Chat-v1.0,8192,3877,0.2551,2048,10,204.80,205.0551,high,"With prompt_tokens=3877, estimated TTFT is abo..."
36,packed_prompt_32768,TinyLlama/TinyLlama-1.1B-Chat-v1.0,32768,3877,0.2551,2048,10,204.80,205.0551,high,"With prompt_tokens=3877, estimated TTFT is abo..."
76,packed_prompt_32768,mistralai/Mistral-7B-Instruct-v0.2,32768,3836,0.2534,2048,10,204.80,205.0534,high,"With prompt_tokens=3836, estimated TTFT is abo..."
56,packed_prompt_8192,mistralai/Mistral-7B-Instruct-v0.2,8192,3836,0.2534,2048,10,204.80,205.0534,high,"With prompt_tokens=3836, estimated TTFT is abo..."
116,packed_prompt_8192,Qwen/Qwen2.5-1.5B-Instruct,8192,3371,0.2348,2048,10,204.80,205.0348,high,"With prompt_tokens=3371, estimated TTFT is abo..."
...,...,...,...,...,...,...,...,...,...,...,...
103,packed_prompt_8192,Qwen/Qwen2.5-1.5B-Instruct,8192,3371,0.2348,128,100,1.28,1.5148,low,"With prompt_tokens=3371, estimated TTFT is abo..."
123,packed_prompt_32768,Qwen/Qwen2.5-1.5B-Instruct,32768,3371,0.2348,128,100,1.28,1.5148,low,"With prompt_tokens=3371, estimated TTFT is abo..."
83,packed_prompt_4096,Qwen/Qwen2.5-1.5B-Instruct,4096,3371,0.2348,128,100,1.28,1.5148,low,"With prompt_tokens=3371, estimated TTFT is abo..."
143,packed_prompt_4096,TinyLlama/TinyLlama-1.1B-Chat-v1.0,4096,2663,0.2065,128,100,1.28,1.4865,low,"With prompt_tokens=2663, estimated TTFT is abo..."


## Summary Findings

These summary values answer the main Step 6 questions:
- which output budget creates the longest decode time
- how tokens/sec changes total latency
- why long outputs are expensive
- when decode dominates total latency
- when prefill dominates total latency

In [5]:
summary = build_decode_summary(df)
summary

{'longest_decode_output_budget': 2048,
 'highest_total_latency_scenario': 'packed_prompt_8192',
 'highest_total_latency_tokenizer': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
 'how_tokens_per_second_changes_latency': {10: 79.597,
  25: 31.981,
  50: 16.109,
  100: 8.173},
 'why_long_outputs_are_expensive': 'Long outputs require more token-by-token generation steps, so decode time keeps growing as output length increases.',
 'why_decode_dominates_for_long_outputs': 'When output length is large, token-by-token decode time can grow much larger than the initial TTFT.',
 'why_prefill_dominates_for_short_outputs': 'When output is short, the up-front cost of reading the input prompt can be a bigger share of total latency.'}

## Create Graphs

These charts help visualize how output length and tokens/sec drive decode time and total latency.

In [6]:
create_decode_graphs(df, FIGURES_DIR)
sorted(path.name for path in FIGURES_DIR.glob('*.png'))

['character_count_vs_token_count.png',
 'context_usage_by_prompt.png',
 'context_used_percent_by_tokenizer.png',
 'decode_load_by_tokens_per_second.png',
 'decode_time_by_output_budget.png',
 'estimated_ttft_by_prompt.png',
 'fits_vs_exceeds_context.png',
 'full_context_usage.png',
 'full_latency_breakdown.png',
 'included_vs_dropped_sections.png',
 'inference_summary.png',
 'output_tokens_vs_decode_time.png',
 'packed_prompt_usage.png',
 'prefill_load_by_context_window.png',
 'prefill_load_by_prompt.png',
 'prompt_tokens_vs_estimated_ttft.png',
 'remaining_tokens_by_prompt.png',
 'strategy_comparison.png',
 'token_budget_by_section.png',
 'token_count_by_category_and_tokenizer.png',
 'tokenizer_comparison_summary.png',
 'tokenizer_latency_comparison.png',
 'tokens_per_word_by_category_and_tokenizer.png',
 'total_latency_prefill_plus_decode.png']

## What the Graphs Show

1. **Decode time by output token budget**
   This shows how decode time grows as you ask for longer outputs.

2. **Output tokens vs decode time**
   This scatter plot shows the direct relationship between output length and decode time under different tokens/sec assumptions.

3. **Total latency: prefill + decode**
   This combines the Step 5 TTFT estimate with decode time so you can see the full simulated latency picture.

4. **Decode time across tokens-per-second values**
   This shows how faster token generation reduces decode time for the same output length.

## Why This Is Only a Simulation

This notebook does **not** measure real decode throughput.

A real decode benchmark would need:
- an actual model loaded
- streaming generation enabled
- generated token counting
- time measured between first token and final token
- tokens/sec calculation
- repeated runs
- average, p50, and p95 reporting

## Step 6 Takeaway

Long outputs increase decode time because the model must generate token by token. Short outputs can make prefill feel more important, while long outputs often make decode dominate total latency.